# 从零实现 VQ-VAE：离散 Codebook、Straight-Through 与重建

VQ-VAE 把 encoder 连续表示映射到最近的离散 embedding，再由 decoder 重建。本 Notebook 手写最近邻量化、straight-through estimator、codebook/commitment loss、perplexity、卷积 encoder/decoder、受控训练和按 code 解码。

8×8 合成图案只验证离散潜变量机制，不代表高分辨率生成质量。真正生成新图还需要在 code 序列上训练先验模型。

In [ ]:
import copy,hashlib,io,json,math,random,warnings
from types import MappingProxyType
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
SEED66=6601
random.seed(SEED66); np.random.seed(SEED66); torch.manual_seed(SEED66); torch.set_num_threads(1)
def canonical66(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))
def sha66(x): return hashlib.sha256(x).hexdigest()
assert torch.get_num_threads()==1

## 1. 图案数据与切分

四类 8×8 图案：竖条、横条、十字和方框，加入小幅独立噪声后 clamp 到 `[0,1]`。train/validation/test 使用不同 seed，标签只用于分层诊断，不进入 VQ-VAE loss。

输入合同 `[B,1,8,8] float32`。制品绑定图案 recipe、split seed、值域与 tensor snapshot。

In [ ]:
PATTERNS66=("vertical","horizontal","cross","box")
def make_images66(count,seed):
    if count%4: raise ValueError("count_must_cover_classes_equally")
    g=torch.Generator().manual_seed(seed); images=[]; labels=[]
    for i in range(count):
        label=i%4; x=torch.zeros(8,8)
        if label in (0,2): x[:,3:5]=1
        if label in (1,2): x[3:5,:]=1
        if label==3: x[1:7,1]=1; x[1:7,6]=1; x[1,1:7]=1; x[6,1:7]=1
        x=(x+.04*torch.randn(8,8,generator=g)).clamp(0,1); images.append(x[None]); labels.append(label)
    return torch.stack(images),torch.tensor(labels)
train_img66,train_lab66=make_images66(256,SEED66+1); val_img66,val_lab66=make_images66(64,SEED66+2); test_img66,test_lab66=make_images66(64,SEED66+3)
assert train_img66.shape==(256,1,8,8) and train_img66.min()>=0 and train_img66.max()<=1
assert torch.equal(make_images66(256,SEED66+1)[0],train_img66) and not torch.equal(train_img66[:64],val_img66)
assert torch.bincount(train_lab66).tolist()==[64]*4

## 2. 最近邻量化与三条梯度路径

对 encoder latent `z_e:[B,D,H,W]`，展平位置并计算到 codebook `E:[K,D]` 的平方距离，索引 $k=\arg\min_j\|z_e-e_j\|^2$。

- codebook loss：$\|sg[z_e]-z_q\|^2$ 更新 embedding；
- commitment：$\beta\|z_e-sg[z_q]\|^2$ 约束 encoder；
- straight-through：$z_{st}=z_e+sg[z_q-z_e]$，前向等于量化值，重建梯度直接传 encoder。

In [ ]:
class VectorQuantizer66(nn.Module):
    def __init__(self,num_codes=16,embedding_dim=8,beta=.25):
        super().__init__()
        if num_codes<2 or embedding_dim<1 or not math.isfinite(beta) or beta<=0: raise ValueError("quantizer_config")
        self.num_codes=num_codes; self.embedding_dim=embedding_dim; self.beta=beta; self.embedding=nn.Embedding(num_codes,embedding_dim); nn.init.uniform_(self.embedding.weight,-1/num_codes,1/num_codes)
    def forward(self,z_e):
        if z_e.ndim!=4 or z_e.shape[1]!=self.embedding_dim or not torch.isfinite(z_e).all(): raise ValueError("latent_contract")
        flat=z_e.permute(0,2,3,1).reshape(-1,self.embedding_dim); emb=self.embedding.weight
        distances=flat.square().sum(1,keepdim=True)+emb.square().sum(1)-2*flat@emb.T; indices=distances.argmin(1)
        z_q=self.embedding(indices).view(z_e.shape[0],z_e.shape[2],z_e.shape[3],self.embedding_dim).permute(0,3,1,2).contiguous()
        codebook=F.mse_loss(z_q,z_e.detach()); commitment=self.beta*F.mse_loss(z_e,z_q.detach()); z_st=z_e+(z_q-z_e).detach()
        counts=torch.bincount(indices,minlength=self.num_codes).float(); probs=counts/counts.sum(); perplexity=torch.exp(-(probs[probs>0]*probs[probs>0].log()).sum())
        return z_st,indices.view(z_e.shape[0],z_e.shape[2],z_e.shape[3]),codebook+commitment,perplexity,(codebook,commitment)
qprobe66=VectorQuantizer66(3,2,.25)
with torch.no_grad(): qprobe66.embedding.weight.copy_(torch.tensor([[0.,0.],[1.,0.],[0.,1.]]))
ze66=torch.tensor([[[[.9]],[[.1]]]],requires_grad=True); zst66,idx66,vql66,perp66,parts66=qprobe66(ze66)
assert idx66.item()==1 and torch.allclose(zst66.flatten(),torch.tensor([1.,0.])) and 1<=perp66<=3
zst66.sum().backward(retain_graph=True); assert torch.equal(ze66.grad,torch.ones_like(ze66))
qprobe66.zero_grad(); ze66.grad=None; vql66.backward(); assert qprobe66.embedding.weight.grad is not None and ze66.grad is not None

## 3. 卷积 Encoder、2×2 离散网格与 Decoder

encoder 两次 stride-2 卷积把 `[B,1,8,8]` 变为 `[B,8,2,2]`，即每张图 4 个 code。decoder 用两次 transposed convolution 恢复 8×8，并以 sigmoid 保证输出值域。

`decode_codes` 允许从 `[B,2,2]` 整数 code 直接生成重建；越界、浮点或错误网格必须拒绝。

In [ ]:
class Encoder66(nn.Module):
    def __init__(self,latent_dim=8): super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,16,4,2,1),nn.ReLU(),nn.Conv2d(16,latent_dim,4,2,1))
    def forward(self,x):
        if x.ndim!=4 or x.shape[0]<1 or x.shape[1:]!=(1,8,8) or not torch.isfinite(x).all() or bool(((x<0)|(x>1)).any()): raise ValueError("image_contract")
        return self.net(x)
class Decoder66(nn.Module):
    def __init__(self,latent_dim=8): super().__init__(); self.net=nn.Sequential(nn.ConvTranspose2d(latent_dim,16,4,2,1),nn.ReLU(),nn.ConvTranspose2d(16,1,4,2,1),nn.Sigmoid())
    def forward(self,z): return self.net(z)
class VQVAE66(nn.Module):
    def __init__(self,latent_dim=8,num_codes=16,beta=.25): super().__init__(); self.encoder=Encoder66(latent_dim); self.quantizer=VectorQuantizer66(num_codes,latent_dim,beta); self.decoder=Decoder66(latent_dim); self.num_codes=num_codes
    def forward(self,x):
        z_e=self.encoder(x); z_st,indices,vq_loss,perplexity,parts=self.quantizer(z_e); recon=self.decoder(z_st)
        if recon.shape!=x.shape or not torch.isfinite(recon).all(): raise ValueError("reconstruction_contract")
        return recon,indices,vq_loss,perplexity,parts
    def decode_codes(self,codes):
        if codes.dtype!=torch.long or codes.ndim!=3 or codes.shape[1:]!=(2,2) or bool(((codes<0)|(codes>=self.num_codes)).any()): raise ValueError("code_grid_contract")
        z=self.quantizer.embedding(codes).permute(0,3,1,2).contiguous(); return self.decoder(z)
model_probe66=VQVAE66(); rec66,codes66,loss66,pp66,_=model_probe66(train_img66[:3])
assert rec66.shape==(3,1,8,8) and codes66.shape==(3,2,2) and loss66.ndim==0
assert torch.allclose(model_probe66.decode_codes(codes66),model_probe66.decoder(model_probe66.quantizer.embedding(codes66).permute(0,3,1,2)),atol=1e-7)

## 4. 重建与 VQ loss 训练

总目标为 `BCE(reconstruction,x)+vq_loss`。固定 batch generator 抽样 240 步；validation 只监控，test 最后报告。记录 code usage/perplexity，避免所有 latent 都塌到一个 code。

小图案容易过拟合，测试重建改善只是实现 smoke test。真实 VQ-VAE 还会调 codebook 更新、EMA、dead-code reset、感知/adversarial loss。

In [ ]:
torch.manual_seed(SEED66); model66=VQVAE66(); opt66=torch.optim.Adam(model66.parameters(),lr=3e-3); bg66=torch.Generator().manual_seed(SEED66+9)
with torch.no_grad(): init_recon66=model66(test_img66)[0]; initial_bce66=float(F.binary_cross_entropy(init_recon66,test_img66))
hist66=[]; val_hist66=[]
for step66 in range(240):
    idx=torch.randint(0,len(train_img66),(64,),generator=bg66); batch=train_img66[idx]; recon,indices,vq_loss,perplexity,parts=model66(batch)
    total=F.binary_cross_entropy(recon,batch)+vq_loss; opt66.zero_grad(set_to_none=True); total.backward(); torch.nn.utils.clip_grad_norm_(model66.parameters(),5.); opt66.step()
    if step66%40==0:
        hist66.append(float(total.detach()))
        with torch.no_grad(): val_hist66.append(float(F.binary_cross_entropy(model66(val_img66)[0],val_img66)))
with torch.no_grad(): test_recon66,test_codes66,test_vq66,test_perp66,_=model66(test_img66); test_bce66=float(F.binary_cross_entropy(test_recon66,test_img66))
usage66=torch.unique(test_codes66).numel()
assert test_bce66<initial_bce66*.55 and usage66>=3 and test_perp66>1.5
assert all(math.isfinite(v) for v in hist66+val_hist66) and val_hist66[-1]<val_hist66[0]
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model66.parameters())
print({"initial_bce":round(initial_bce66,4),"test_bce":round(test_bce66,4),"codes_used":usage66,"perplexity":round(float(test_perp66),3)})

## 5. Code 语义、扰动与失败边界

同类图案不保证使用完全相同 code，因为 VQ-VAE 没有类别监督；但重建应有限且 code 网格可复现。改变一个 code 后输出应变化，证明 decoder 真正依赖离散 latent。

Perplexity 是使用均匀度而非生成质量；高 perplexity 可能只是噪声编码，低 perplexity 可能是数据本身简单。

In [ ]:
with torch.no_grad():
    base_codes66=test_codes66[:1].clone(); changed_codes66=base_codes66.clone(); changed_codes66[0,0,0]=(changed_codes66[0,0,0]+1)%model66.num_codes
    base_decode66=model66.decode_codes(base_codes66); changed_decode66=model66.decode_codes(changed_codes66)
assert not torch.allclose(base_decode66,changed_decode66) and torch.isfinite(changed_decode66).all()
try: model66.decode_codes(base_codes66.float()); raise AssertionError("float codes accepted")
except ValueError as e: assert str(e)=="code_grid_contract"
try: model66(torch.full((1,1,8,8),2.)); raise AssertionError("out-of-range image accepted")
except ValueError as e: assert str(e)=="image_contract"

### 5.1 Codebook 使用率、类别切片与最近邻 oracle

单个 perplexity 不足以解释 codebook。这里同时查看每类图案的重建 BCE、所有 code 的计数与概率，并从 encoder latent 手工重算平方距离和最近邻索引。这样能发现维度 permute 错误、距离公式符号错误，以及“整体 perplexity 正常但某一类完全塌缩”的情况。

这些类别切片只用于离线诊断，标签仍不参与训练。真实无标签数据可以按来源、时间段、分辨率或业务切片替代类别；若长期存在 dead code，可考虑 EMA 更新、重置低频 embedding 或调整 commitment 权重，而不是只追求更大的 codebook。

In [ ]:
per_image_bce66=F.binary_cross_entropy(test_recon66,test_img66,reduction="none").flatten(1).mean(1)
class_bce66=torch.stack([per_image_bce66[test_lab66==label].mean() for label in range(len(PATTERNS66))])
counts66=torch.bincount(test_codes66.flatten(),minlength=model66.num_codes).float(); probs66=counts66/counts66.sum()
recomputed_perplexity66=torch.exp(-(probs66[probs66>0]*probs66[probs66>0].log()).sum())
assert class_bce66.shape==(4,) and torch.isfinite(class_bce66).all()
assert counts66.sum().item()==len(test_img66)*4 and int((counts66>0).sum())==usage66
assert torch.allclose(recomputed_perplexity66,test_perp66,atol=1e-6)
with torch.no_grad():
    audit_ze66=model66.encoder(test_img66[:3]); audit_flat66=audit_ze66.permute(0,2,3,1).reshape(-1,8); audit_emb66=model66.quantizer.embedding.weight
    audit_distance66=audit_flat66.square().sum(1,keepdim=True)+audit_emb66.square().sum(1)-2*audit_flat66@audit_emb66.T
    audit_index66=audit_distance66.argmin(1).view(3,2,2)
assert torch.equal(audit_index66,test_codes66[:3])
assert bool((audit_distance66>=-1e-6).all()) and torch.isfinite(audit_distance66).all()
assert torch.equal(model66.decode_codes(test_codes66[:2]),model66.decode_codes(test_codes66[:2]))

## 6. Published VQ-VAE

manifest 绑定图案/seed/split snapshot、输入值域、encoder/decoder shape、codebook 大小/维度、beta、optimizer 和训练步数。Published wrapper 提供 `reconstruct` 与 `decode_codes`，仍执行输入合同。

包外 registry 是发布信任锚；整体替换 codebook/decoder 并重算内部 hash 仍被拒绝。

In [ ]:
def th66(t):
    v=t.detach().cpu().contiguous(); return sha66(str(v.dtype).encode()+canonical66(list(v.shape)).encode()+v.numpy().tobytes())
def sd66(state):
    h=hashlib.sha256()
    for k,v in sorted(state.items()): h.update(k.encode()); h.update(th66(v).encode())
    return h.hexdigest()
manifest66={"artifact_id":"vqvae-patterns-v1","version":1,"model_config":{"latent_dim":8,"num_codes":16,"beta":.25},"data":{"patterns":list(PATTERNS66),"splits":{"train":[256,SEED66+1,th66(train_img66),th66(train_lab66)],"val":[64,SEED66+2,th66(val_img66),th66(val_lab66)],"test":[64,SEED66+3,th66(test_img66),th66(test_lab66)]}},"preprocess":{"shape":[1,8,8],"range":[0.,1.]},"training":{"seed":SEED66,"optimizer":"Adam","steps":240,"batch":64,"lr":.003,"grad_clip":5.,"validation_interval":40}}
def pkg66(model,m):
    b=io.BytesIO(); torch.save(model.state_dict(),b); raw=b.getvalue(); state=torch.load(io.BytesIO(raw),map_location="cpu",weights_only=True); ms=sha66(canonical66(m).encode()); ss=sd66(state); rs=sha66(raw); bd=sha66(canonical66([ms,ss,rs]).encode()); return {"manifest":copy.deepcopy(m),"manifest_sha":ms,"state_bytes":raw,"state_digest":ss,"state_bytes_sha":rs,"bundle_digest":bd}
package66=pkg66(model66,manifest66); REG66=MappingProxyType({("vqvae-patterns-v1",1):package66["bundle_digest"]})
class PublishedVQVAE66:
    def __init__(self,model): self._model=model
    @torch.no_grad()
    def reconstruct(self,images): return self._model(torch.as_tensor(images,dtype=torch.float32))[0]
    @torch.no_grad()
    def decode_codes(self,codes): return self._model.decode_codes(torch.as_tensor(codes))
def load66(pkg):
    m=pkg["manifest"]; key=(m.get("artifact_id"),m.get("version"))
    current_ms=sha66(canonical66(m).encode()); current_rs=sha66(pkg["state_bytes"])
    if m!=manifest66 or current_ms!=pkg["manifest_sha"] or current_rs!=pkg["state_bytes_sha"]: raise RuntimeError("package_contract")
    for name,(count,seed,image_digest,label_digest) in m["data"]["splits"].items():
        regen_image,regen_label=make_images66(count,seed)
        if th66(regen_image)!=image_digest or th66(regen_label)!=label_digest: raise RuntimeError("data_snapshot_contract")
    state=torch.load(io.BytesIO(pkg["state_bytes"]),map_location="cpu",weights_only=True)
    current_sd=sd66(state)
    if current_sd!=pkg["state_digest"]: raise RuntimeError("state_contract")
    current_bundle=sha66(canonical66([current_ms,current_sd,current_rs]).encode())
    if pkg.get("bundle_digest")!=current_bundle: raise RuntimeError("bundle_contract")
    if REG66.get(key)!=current_bundle: raise RuntimeError("publisher_registry_rejected")
    model=VQVAE66(**m["model_config"]); model.load_state_dict(state); model.eval(); return PublishedVQVAE66(model)
pub66=load66(package66); assert torch.allclose(pub66.reconstruct(test_img66[:2]),test_recon66[:2],atol=1e-7)
forged66=pkg66(VQVAE66(),manifest66)
try: load66(forged66); raise AssertionError("re-signed VQVAE accepted")
except RuntimeError as e: assert str(e)=="publisher_registry_rejected"
forged_old_bundle66=copy.deepcopy(forged66); forged_old_bundle66["bundle_digest"]=package66["bundle_digest"]
try: load66(forged_old_bundle66); raise AssertionError("forged VQVAE with old bundle accepted")
except RuntimeError as e: assert str(e)=="bundle_contract"
assert isinstance(REG66,MappingProxyType)

### 6.1 推理 API 与 fail-closed 测试

重建接口与 code 解码接口承担不同合同：前者接受 `[B,1,8,8]` 浮点图像，后者接受 `[B,2,2]` 的整数索引。服务不能悄悄 clamp 非法像素或浮点 code，因为那会隐藏上游 schema 漂移；也不能只校验 checkpoint 能被 `torch.load`，还必须验证发布身份、manifest、原始字节和逐 tensor 摘要。

下面覆盖确定性、错误 shape、NaN、越界 code、manifest 篡改与权重字节破坏。上线后还应限制 batch 大小，并对 code 使用率、重建误差分位数和输入值域违规分别告警。

In [ ]:
assert pub66._model.training is False and torch.isfinite(pub66.reconstruct(test_img66[:2])).all()
assert torch.equal(pub66.reconstruct(test_img66[:2]),pub66.reconstruct(test_img66[:2]))
assert torch.allclose(pub66.decode_codes(test_codes66[:2]),model66.decode_codes(test_codes66[:2]),atol=1e-7)
try: pub66.reconstruct(torch.zeros(1,1,7,8)); raise AssertionError("wrong image shape accepted")
except ValueError as e: assert str(e)=="image_contract"
try: pub66.reconstruct(torch.full((1,1,8,8),float("nan"))); raise AssertionError("NaN image accepted")
except ValueError as e: assert str(e)=="image_contract"
try: pub66.reconstruct(torch.empty(0,1,8,8)); raise AssertionError("empty image batch accepted")
except ValueError as e: assert str(e)=="image_contract"
invalid_codes66=test_codes66[:1].clone(); invalid_codes66[0,0,0]=model66.num_codes
try: pub66.decode_codes(invalid_codes66); raise AssertionError("out-of-range code accepted")
except ValueError as e: assert str(e)=="code_grid_contract"
tampered66=copy.deepcopy(package66); tampered66["manifest"]["model_config"]["num_codes"]+=1
try: load66(tampered66); raise AssertionError("tampered manifest accepted")
except RuntimeError as e: assert str(e)=="package_contract"
damaged66=copy.deepcopy(package66); damaged66["state_bytes"]=damaged66["state_bytes"]+b"x"
try: load66(damaged66); raise AssertionError("damaged state bytes accepted")
except RuntimeError as e: assert str(e)=="package_contract"
state_copy66={k:v.clone() for k,v in model66.state_dict().items()}; first_key66=next(iter(state_copy66)); state_copy66[first_key66].view(-1)[0]+=1
assert sd66(state_copy66)!=sd66(model66.state_dict())

## 7. 复杂度、失败模式与来源

距离计算成本约 $O(BHWKD)$，大 codebook 会占显存；可用分块、EMA 或近邻索引优化。常见错误：codebook/commitment stop-gradient 写反；忘 straight-through；code 越界；只看 perplexity；decoder 接连续 latent 绕过量化；发布时丢失 codebook 顺序。

- van den Oord et al., [Neural Discrete Representation Learning](https://arxiv.org/abs/1711.00937), NeurIPS 2017。
- Razavi et al., [VQ-VAE-2](https://arxiv.org/abs/1906.00446), NeurIPS 2019。
- Esser et al., [Taming Transformers](https://arxiv.org/abs/2012.09841)，感知/对抗式离散 autoencoder 背景。